# L'effet domino dans le tunnel RER B / RER D

**Projet Python pour la data science — ENSAE 2A**

## Problématique

Entre *Châtelet-Les Halles* et *Gare du Nord*, le RER B et le RER D circulent sur **les mêmes voies**. Les usagers ont l'impression qu'un incident sur l'une des deux lignes se propage rapidement à l'autre.

**Hypothèse :** un incident sur le RER B (ou D) se propage à l'autre ligne à cause du tronçon commun, ce qui rend la régulation plus difficile et les retards plus persistants.

**Point de comparaison :** le **RER A**, dont le tronçon central (*Charles de Gaulle-Étoile* – *Nation*) est très fréquenté mais **ne partage pas ses voies** avec une autre ligne de RER. Il sert de référence (placebo).

## Plan

0. Préparation de l'environnement
1. Données : collecte, nettoyage et description
2. Axe 1 — Constat : B et D sont-ils plus en retard que A ?
3. Axe 2 — Contagion : les retards de B et D sont-ils liés ?
4. Axe 3 — Détection automatique des principaux incidents
5. Conclusion et limites

> Le code des analyses est dans le dossier `src/` (un fichier par axe). Ce notebook les exécute dans l'ordre et en interprète les résultats.

## 0. Préparation

À lancer depuis la racine du dépôt avec *Restart & Run All*. Au premier lancement, les données sont téléchargées depuis le S3 public du SSP Cloud (quelques minutes).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## 1. Données

- **Source** : API temps réel PRIM d'Île-de-France Mobilités (passages des RER A, B, D et messages d'incidents), collectée toutes les 2 minutes par notre collecteur (`bot_collect/`) depuis le 02/10/2026 et stockée sur un S3 public.
- **Nettoyage** (`src/lecture.py`, `src/nettoyage.py`) : une ligne = un arrêt d'un train dans une gare du **tronçon central**, horaires en heure de Paris, retards aberrants retirés, variables d'analyse créées (retard, retard ajouté depuis l'arrêt précédent, suppression, tunnel, tranche horaire).

Détails dans `docs/pipeline_donnees.md`.

In [ ]:
from src.nettoyage import charger_donnees_propres

# Télécharge (si besoin) et nettoie les données, puis les charge
df_a, df_b, df_d, df_inc = charger_donnees_propres()

In [ ]:
lignes = {"RER A": df_a, "RER B": df_b, "RER D": df_d}
apercu = pd.DataFrame({
    "Début": [d["date"].min().date() for d in lignes.values()],
    "Fin": [d["date"].max().date() for d in lignes.values()],
    "Gares du tronçon central": [d["stop_name"].nunique() for d in lignes.values()],
    "Trains": [len(d.drop_duplicates(["date", "journey_id"])) for d in lignes.values()],
    "Arrêts observés": [len(d) for d in lignes.values()],
}, index=lignes.keys())
apercu

In [ ]:
# Une ligne du tableau = un arrêt d'un train dans une gare du tronçon central
colonnes = ["date", "journey_id", "direction", "stop_name", "aimed_time", "expected_time",
            "retard_min", "delay_added_sec", "est_supprime", "tranche_horaire"]
df_b[colonnes].head()

In [ ]:
print(f"{len(df_inc)} messages d'incidents collectés")
df_inc[["line", "first_seen", "short_message"]].head()

Le D compte moins d'observations (5 gares centrales, collecte à partir du 07/10) : on compare donc toujours des moyennes ou des proportions.

## 2. Axe 1 — B et D sont-ils plus en retard que A ?

`src/Axe1_constat_naif.py` compare les trois lignes : distribution des retards, retard pris dans le tunnel, heure de pointe, suppressions et retard jour par jour.

In [ ]:
%run src/Axe1_constat_naif.py

**À retenir** (données du 02/10 au 26/11/2026) :

- B et D sont **2 à 3 fois plus en retard** que A, en moyenne comme à Châtelet, et même un jour ordinaire. Le 95e centile atteint environ 13 min contre 7 pour A.
- **Le retard se crée dans le tunnel**, surtout à sa sortie : B et D y perdent 27 à 41 s par arrêt quand A en rattrape 11.
- La régulation **coupe les trains** : 2 à 3,5 % des B et D sont limités en parcours, contre 0,2 % des A.

## 3. Axe 2 — Les retards de B et D sont-ils liés ?

`src/axe2_contagion.py` travaille sur le retard moyen par quart d'heure à Châtelet : corrélations, décalage dans le temps (`.shift()`), retard d'une ligne selon l'état de l'autre et temps de retour à la normale. Le RER A sert de placebo.

In [ ]:
%run src/axe2_contagion.py

**À retenir** :

- Les retards de B et D **évoluent ensemble** (corrélation 0,26 contre 0,08 avec A), même sans l'effet commun de l'heure de pointe (0,22 contre 0,02).
- Le retard de B se retrouve sur D **15 minutes plus tard**, et l'effet joue dans les deux sens.
- Quand B est perturbé, le retard de D passe de 2,3 à **7,4 min** ; A reste vers 1 min.
- B est le plus lent à se remettre : **150 min** en médiane après un pic, contre environ 70 pour A et D.

## 4. Axe 3 — Les principaux incidents

`src/axe3_incidents.py` détecte automatiquement les épisodes de crise dans le tunnel (gros retard, chute du nombre de trains ou suppressions massives), les croise avec les messages officiels, puis zoome sur le plus important.

In [ ]:
%run src/axe3_incidents.py

**À retenir** :

- Les 8 plus gros incidents durent **de 2 h à près de 8 h** ; au pire, le nombre de trains dans le tunnel tombe à 15 % de la normale.
- **6 sur 8 touchent B et D ensemble**, alors que A reste presque toujours sous 5 min. Quelques incidents ne concernent qu'une ligne (ex. incident SNCF du 15/10 sur le D seul).
- Le 21/10, le nombre de trains est divisé par deux, D dépasse 50 min de retard puis B suit et reste perturbé plus longtemps ; le diagramme de Marey montre les B coupés avant le tunnel.

## 5. Conclusion

### Réponse à la problématique

Les trois axes vont dans le même sens :

1. **Constat (axe 1)** : B et D sont plus en retard que A, surtout lors des perturbations graves, et ce retard se crée dans le tunnel commun, particulièrement à sa sortie.
2. **Contagion (axe 2)** : les retards de B et D évoluent ensemble, le retard de B précède celui de D d'environ 15 minutes, et l'effet joue dans les deux sens. Le RER A, utilisé comme placebo, ne montre pas ce lien. B met aussi plus longtemps à revenir à la normale.
3. **Incidents (axe 3)** : 6 des 8 principaux épisodes de crise touchent B et D ensemble, avec une forte chute du nombre de trains dans le tunnel, alors que A reste stable. Quelques incidents restent toutefois propres à une seule ligne.

L'hypothèse d'un **effet domino dans le tunnel B / D** est donc **confortée par les données**.

### Limites

- **Corrélation n'est pas causalité** : B et D diffèrent aussi de A par leur longueur, leurs branches et leur exploitation (deux opérateurs, RATP et SNCF).
- **Les liens mesurés restent modérés** : une corrélation d'environ 0,26 montre une tendance, pas une transmission systématique de chaque retard.
- **Les données temps réel sont imparfaites** : la collecte du RER D commence plus tard, il existe des trous de collecte, et le retard des trains supprimés n'est pas renseigné.
- **Plusieurs seuils sont des choix** : 3, 5 et 10 minutes, fenêtres de 15 minutes, critères de crise. Les conclusions devraient être vérifiées avec d'autres valeurs.
- **La période est courte** (un peu moins de deux mois, en automne) et s'allonge à chaque mise à jour des données : les chiffres peuvent légèrement évoluer.

### Pistes

- Prolonger la collecte sur plusieurs saisons (hiver, vacances scolaires) et croiser les retards avec les données de météo ou de fréquentation.

### Usage de l'IA générative

Nous avons utilisé une IA générative pour nous aider à structurer et simplifier une partie du code et la mise en forme des graphiques. Les analyses et leur interprétation ont été réalisées par l'équipe.